In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta
import os

# Create folders
os.makedirs("data/internal", exist_ok=True)
os.makedirs("data/external", exist_ok=True)

def generate_transactions(n=100):

    base_time = datetime.now()

    internal_data = []
    gateway_data = []
    bank_data = []

    for i in range(n):

        txn_id = f"txn_{i+1}"

        amount = round(random.uniform(1000, 500000), 2)

        customer_id = f"user_{random.randint(1, 50)}"

        timestamp = base_time + timedelta(seconds=i)

        # -------------------------
        # INTERNAL SYSTEM RECORD
        # -------------------------

        internal_data.append({
            "transaction_id": txn_id,
            "customer_id": customer_id,
            "amount": amount,
            "status": "SUCCESS",
            "timestamp": timestamp
        })

        # -------------------------
        # GATEWAY RECORD
        # Simulate mismatches
        # -------------------------

        if random.random() > 0.1:

            gateway_data.append({
                "transaction_id": txn_id,
                "gateway_status": "SUCCESS",

                # 10% chance amount mismatch
                "amount": amount if random.random() > 0.1 else round(amount * 0.9, 2),

                "timestamp": timestamp
            })

        # -------------------------
        # BANK SETTLEMENT RECORD
        # Simulate missing settlements
        # -------------------------

        if random.random() > 0.2:

            bank_data.append({
                "transaction_id": txn_id,
                "settlement_status": "SETTLED",
                "amount": amount,
                "timestamp": timestamp
            })

    # Save files

    pd.DataFrame(internal_data).to_csv(
        "data/internal/internal_transactions.csv",
        index=False
    )

    pd.DataFrame(gateway_data).to_csv(
        "data/external/gateway_transactions.csv",
        index=False
    )

    pd.DataFrame(bank_data).to_csv(
        "data/external/bank_settlements.csv",
        index=False
    )

    print("✅ Data generation complete!")

# Run script
generate_transactions(200)

✅ Data generation complete!


In [2]:
import pandas as pd
import os

# Paths
input_path = "data/internal/internal_transactions.csv"

output_path = "data/processed/ledger_entries.csv"

# Ensure processed folder exists
os.makedirs("data/processed", exist_ok=True)

def create_ledger_entries():

    print("Creating ledger entries...\n")

    df = pd.read_csv(input_path)

    ledger_entries = []

    for _, row in df.iterrows():

        txn_id = row["transaction_id"]

        amount = row["amount"]

        customer_id = row["customer_id"]

        timestamp = row["timestamp"]

        # -------------------------
        # DEBIT ENTRY
        # -------------------------

        ledger_entries.append({
            "transaction_id": txn_id,
            "account": f"{customer_id}_wallet",
            "entry_type": "DEBIT",
            "amount": amount,
            "timestamp": timestamp
        })

        # -------------------------
        # CREDIT ENTRY
        # -------------------------

        ledger_entries.append({
            "transaction_id": txn_id,
            "account": "company_settlement_account",
            "entry_type": "CREDIT",
            "amount": amount,
            "timestamp": timestamp
        })

    ledger_df = pd.DataFrame(ledger_entries)

    ledger_df.to_csv(output_path, index=False)

    print("✅ Ledger entries created!")

# Run script
create_ledger_entries()

Creating ledger entries...

✅ Ledger entries created!


In [3]:
import pandas as pd
import os

# Input files
internal_path = "data/internal/internal_transactions.csv"

gateway_path = "data/external/gateway_transactions.csv"

bank_path = "data/external/bank_settlements.csv"

# Output file
output_path = "data/processed/reconciliation_results.csv"

# Ensure processed folder exists
os.makedirs("data/processed", exist_ok=True)

def reconcile_transactions():

    print("Running reconciliation...\n")

    # Load datasets
    internal_df = pd.read_csv(internal_path)

    gateway_df = pd.read_csv(gateway_path)

    bank_df = pd.read_csv(bank_path)

    reconciliation_results = []

    for _, txn in internal_df.iterrows():

        txn_id = txn["transaction_id"]

        internal_amount = txn["amount"]

        # -------------------------
        # MATCH IN GATEWAY
        # -------------------------

        gateway_match = gateway_df[
            gateway_df["transaction_id"] == txn_id
        ]

        # -------------------------
        # MATCH IN BANK
        # -------------------------

        bank_match = bank_df[
            bank_df["transaction_id"] == txn_id
        ]

        issue = "MATCHED"

        # Missing in gateway
        if gateway_match.empty:
            issue = "MISSING_IN_GATEWAY"

        # Missing in bank
        elif bank_match.empty:
            issue = "MISSING_IN_BANK"

        else:

            gateway_amount = gateway_match.iloc[0]["amount"]

            # Amount mismatch
            if internal_amount != gateway_amount:
                issue = "AMOUNT_MISMATCH"

        reconciliation_results.append({
            "transaction_id": txn_id,
            "internal_amount": internal_amount,
            "reconciliation_status": issue
        })

    results_df = pd.DataFrame(reconciliation_results)

    results_df.to_csv(output_path, index=False)

    print("✅ Reconciliation complete!")

# Run reconciliation
reconcile_transactions()

Running reconciliation...

✅ Reconciliation complete!


In [4]:
import pandas as pd
import os

# Input reconciliation results
input_path = "data/processed/reconciliation_results.csv"

# Output alerts file
output_path = "data/processed/discrepancy_alerts.csv"

# Ensure processed folder exists
os.makedirs("data/processed", exist_ok=True)

def generate_alerts():

    print("Generating discrepancy alerts...\n")

    # Load reconciliation results
    df = pd.read_csv(input_path)

    # Filter only failed reconciliations
    alerts_df = df[
        df["reconciliation_status"] != "MATCHED"
    ]

    # Save alerts
    alerts_df.to_csv(output_path, index=False)

    print(f"🚨 {len(alerts_df)} discrepancy alerts generated!")

# Run alerts
generate_alerts()

Generating discrepancy alerts...

🚨 78 discrepancy alerts generated!
